# 📅 Day 19 — Mock Interview Practice

**Week 3 · Day 5 · Level: ADVANCED (14 ADVANCED + 4 HARD)**

Week 3 continues: **product-company level** questions — complex end-to-end scenarios, engine internals, and system-design judgment. 18 questions across 8 categories. Day 19 themes: **multi-touch attribution, fuzzy entity resolution with levenshtein, k-way merge of sorted streams, consistent hashing rings, arbitrary stateful streaming (`flatMapGroupsWithState`), stream-to-stream joins, Spark Connect & the UDF serialization tax, sort-merge vs shuffled-hash joins, zero-downtime schema migration, DEEP CLONE disaster recovery, checkpoint-safe streaming upgrades, event sourcing vs state tables, feature-store point-in-time correctness, Snowflake Cortex GenAI pipelines, event tables, dbt + Databricks orchestration, pytest/chispa pipeline testing, and designing a full observability stack** — all fresh vs days 1-18.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED / HARD |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED / HARD |
| 5 | ETL — Failures, Late Data, Schema Handling, Fault Tolerance | Q9-Q11 | ADVANCED / HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED / HARD |

**How to use**: attempt each answer out loud in ≤ 4 minutes BEFORE opening the solution. Every answer must name at least one trade-off. Day 20 continues Week 3; days 22+ move to FAANG / top product-company level.

---

## Q1 — SQL Advanced: Multi-Touch Attribution — First/Last/Linear/Position-Based [ADVANCED]

**Interviewer (Google):** Marketing spent across email, social, search, and display, converting three users for a total of **$800**. Which channel deserves credit? Your CMO wants ONE query comparing four attribution models side by side.

The `touches` table logs every pre-conversion touchpoint per user (one conversion per user; `revenue` is that user's conversion revenue repeated on each touch row).

1. Compute, per channel, attributed revenue under all four models at once:
   * **First-touch** — 100% to the earliest touch
   * **Last-touch** — 100% to the last touch
   * **Linear** — equal credit to every touch
   * **Position-based (U-shaped)** — 40% first, 40% last, the middle 20% split evenly across middle touches (single-touch journeys get 100%)
2. Use window functions (`ROW_NUMBER`, `COUNT OVER`) — **no self-joins**.
3. Prove revenue is fully allocated: every model's column total must equal $800.

**Follow-ups:** why does last-touch systematically starve upper-funnel channels? How would you extend to multiple conversions per user (journey = touches since the previous conversion)? Which model would you defend to a CFO, and why?

In [0]:
# Solution Q1 — one query, four attribution models, window functions only
spark.createDataFrame([
    ("u1", "email",   "2026-01-01 09:00", 400.0),
    ("u1", "social",  "2026-01-02 10:00", 400.0),
    ("u1", "search",  "2026-01-03 11:00", 400.0),
    ("u1", "display", "2026-01-04 12:00", 400.0),
    ("u2", "social",  "2026-01-01 08:00", 300.0),
    ("u2", "social",  "2026-01-02 09:30", 300.0),
    ("u2", "email",   "2026-01-03 10:00", 300.0),
    ("u3", "display", "2026-01-05 09:00", 100.0),
], "user_id string, channel string, event_time string, revenue double").createOrReplaceTempView("q1_touches")

res = spark.sql("""
WITH journey AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time, channel) AS rn,
         COUNT(*)     OVER (PARTITION BY user_id)                            AS n
  FROM q1_touches
), weighted AS (
  SELECT user_id, channel, revenue,
    CASE WHEN n = 1 THEN 1.0                  -- single touch takes everything
         WHEN rn = 1 THEN 0.40                -- first 40%
         WHEN rn = n THEN 0.40                -- last 40%
         ELSE 0.20 / (n - 2) END AS pos_w,    -- middle 20% split evenly
    CASE WHEN rn = 1 THEN 1.0 ELSE 0.0 END AS first_w,
    CASE WHEN rn = n THEN 1.0 ELSE 0.0 END AS last_w,
    1.0 / n AS linear_w
  FROM journey
)
SELECT channel,
       ROUND(SUM(revenue * first_w),  2) AS first_touch,
       ROUND(SUM(revenue * last_w),   2) AS last_touch,
       ROUND(SUM(revenue * linear_w), 2) AS linear,
       ROUND(SUM(revenue * pos_w),    2) AS position_based
FROM weighted
GROUP BY channel
ORDER BY channel
""")
res.show()

q1 = {r["channel"]: r for r in res.collect()}
# 1) every model fully allocates the $800 of conversion revenue
for col in ("first_touch", "last_touch", "linear", "position_based"):
    total = sum(r[col] for r in q1.values())
    assert abs(total - 800.0) < 0.01, (col, total)
# 2) spot-check the models against hand math
assert abs(q1["email"]["first_touch"] - 400.0) < 0.01      # u1's first touch
assert abs(q1["display"]["last_touch"] - 500.0) < 0.01    # u1+u3's last touch
assert abs(q1["social"]["position_based"] - 220.0) < 0.01
assert abs(q1["search"]["position_based"] - 40.0) < 0.01
print("Q1 OK — 4 attribution models agree on total revenue")
print("""
[SAY THIS OUT LOUD]
- Last-touch starves upper-funnel channels (search/social here): a channel that ASSISTS the
  journey never gets credit. Position-based protects both ends of the funnel.
- Linear is the CFO-defensible default: zero parameters, fully auditable.
- Multi-conversion extension: journey = touches BETWEEN conversions -> derive a journey id with
  gap-based sessionization (islands), then re-run the same weighting per journey.
""")

## Q2 — SQL Advanced: Fuzzy Entity Resolution — `levenshtein` + Blocking [ADVANCED]

**Interviewer (Salesforce):** The CRM has duplicate customers created by name typos (`"Acme Corp"` vs `"Acme  Corp"` vs `"Acmee Corp"`). Marketing counts are inflated. Find the duplicate clusters and pick a **golden record** for each.

1. Why is comparing **all n² pairs** a non-starter at 10M accounts? What is **blocking**, and how much does a `substr(name,1,4) + city` block cut the search space?
2. Write ONE query: self-join within a block, keep pairs with `levenshtein(name_a, name_b) <= 2`, dedupe the (a,b)/(b,a) mirror, then pick the golden record per cluster with survivorship rules (most complete + most recent).
3. Trade-offs: what does a **tighter** block do to precision vs recall? How do false merges vs missed merges hurt differently?

In [0]:
# Solution Q2 — blocking + levenshtein pairing + golden-record survivorship
from pyspark.sql import functions as F
from pyspark.sql.window import Window

accounts = spark.createDataFrame([
    (1, "Acme Corp",      "Austin",  "billing@acme.com",   "2026-10-20"),
    (2, "Acme  Corp",     "Austin",  None,                  "2026-10-25"),   # dup (typo+spaces)
    (3, "Acmee Corp",      "Austin",  "ap@acme.com",        "2026-10-30"),   # dup (one edit)
    (4, "Acme Corp",       "Denver",  "info@acme.io",       "2026-10-21"),   # DIFFERENT city -> not merged
    (5, "Globex Ltd",     "Austin",  "hello@globex.com",   "2026-10-22"),
    (6, "Globeex Ltd",    "Austin",  None,                  "2026-10-28"),   # dup
], ["acct_id", "name", "city", "email", "updated_at"])

accounts = accounts.withColumn("updated_at", F.col("updated_at").cast("date"))

# --- Step 1: candidate pairs within a block (name-prefix + city) ---
a = accounts.withColumnRenamed("acct_id", "id_a").withColumnRenamed("name", "name_a") \
     .withColumn("block", F.concat(F.expr("substr(name_a, 1, 4)"), F.lit("|"), F.col("city")))
b = accounts.withColumnRenamed("acct_id", "id_b").withColumnRenamed("name", "name_b") \
     .withColumn("block", F.concat(F.expr("substr(name_b, 1, 4)"), F.lit("|"), F.col("city")))

pairs = (a.alias("a").join(b.alias("b"), (F.col("a.block") == F.col("b.block")) & (F.col("a.id_a") < F.col("b.id_b")), "inner")
         .withColumn("edit_distance", F.levenshtein(F.col("a.name_a"), F.col("b.name_b"))))

candidates = pairs.filter("edit_distance <= 2").select("id_a", "id_b", "edit_distance")
candidates.show()

# --- Step 2: golden record via survivorship (completeness first, recency second) ---
# A duplicate can appear in SEVERAL candidate pairs (acct 3 matches both 1 and 2) ->
# collapse to ONE canonical parent per account first, or the left join fans out rows.
parents = candidates.groupBy("id_b").agg(F.min("id_a").alias("parent"))
# NOTE: min-parent handles star-shaped clusters; long transitive chains need full
# closure (iterative label propagation) or the union-find from Q4 - say so in the room.
w = Window.partitionBy("cluster").orderBy(F.col("emails_present").desc(), F.col("updated_at").desc())
clustered = (accounts
    .join(parents, accounts.acct_id == parents.id_b, "left")
    .withColumn("cluster", F.coalesce("parent", "acct_id"))         # union-find-lite: dup points at its parent
    .withColumn("emails_present", F.col("email").isNotNull().cast("int")))
golden = (clustered.withColumn("rank", F.row_number().over(w))
    .filter("rank = 1")
    .select(F.col("cluster").alias("golden_id"), "acct_id", "name", "city", "email"))

golden.orderBy("golden_id", "acct_id").show(truncate=False)

n_golden = golden.select("golden_id").distinct().count()
assert n_golden == 3, f"expected 3 customers (Acme-Austin, Acme-Denver, Globex), got {n_golden}"
assert candidates.filter("id_a=1 AND id_b=4").count() == 0  # city mismatch must NOT merge
print("Q2 OK — 6 rows -> 3 golden customers; Denver kept separate; survivorship correct")

## Q3 — Python: K-Way Merge of Sorted Streams [ADVANCED]

**Interviewer (Meta):** You are ingesting k time-ordered event logs — one per shard, each an **iterator** that may be far larger than memory. Downstream needs ONE sorted stream. Build it without ever materializing all the data.

1. Implement `merge_sorted(streams)` as a **lazy generator**: a min-heap holds ONE element per active stream; every pull is O(log k); memory is O(k).
2. Handle exhausted and empty streams via a **sentinel** — and explain why `None` is unsafe as the sentinel (what if a stream legitimately yields `None`?).
3. Ties must break deterministically by stream index (stability earns points).

**Follow-ups:** how does your hand-rolled version relate to `heapq.merge`? Why is this the core loop of **external merge sort** (k sorted runs, bounded RAM)? How would you turn it into a "top-k across shards" that never materializes the merged list?

In [0]:
# Solution Q3 — lazy k-way merge with a min-heap (sentinel, not None)
import heapq

SENTINEL = object()   # unique object: can never collide with real data (None CAN)

def merge_sorted(streams):
    """Lazily merge k sorted iterables. O(log k) per element, O(k) memory."""
    heap = []
    iters = [iter(s) for s in streams]
    for i, it in enumerate(iters):
        first = next(it, SENTINEL)
        if first is not SENTINEL:                    # skip empty/exhausted streams
            heapq.heappush(heap, (first, i))           # (value, stream idx) -> stable ties
    while heap:
        val, i = heapq.heappop(heap)
        yield val                                    # lazy: one element at a time
        nxt = next(iters[i], SENTINEL)
        if nxt is not SENTINEL:
            heapq.heappush(heap, (nxt, i))

s1, s2, s3, s4 = [1, 4, 7, 10], [2, 2, 5], [], [0, 3, 100]
merged = list(merge_sorted([s1, s2, s3, s4]))
assert merged == sorted(s1 + s2 + s3 + s4) == [0, 1, 2, 2, 3, 4, 5, 7, 10, 100]

gen = merge_sorted([[10, 20], [5, 15]])              # laziness proof: nothing computed eagerly
assert [next(gen) for _ in range(4)] == [5, 10, 15, 20]
try:
    next(gen)
    raise AssertionError("generator should be exhausted")
except StopIteration:
    pass

print("Q3 OK — merged:", merged)
print("""
[SAY THIS OUT LOUD]
- The None-as-sentinel bug: a stream that legitimately yields None would END that stream early.
  A module-level object() can never appear in the data.
- heapq.merge is exactly this loop in the stdlib (pass key= for non-comparable payloads).
- External merge sort: spill k sorted runs to disk, then this heap streams them back in a
  RAM-bounded merge pass. Top-k across shards = same heap, just stop after k pops.
""")

## Q4 — Python: Consistent Hashing — Ring with Virtual Nodes [HARD]

**Interviewer (Amazon):** You shard a cache across 3 nodes using `hash(key) % 3`. A 4th node is added — what fraction of your cache is invalidated? (Answer: ~75%.) Design the fix Dynamo, Cassandra, and Kafka partitioners use.

Build `ConsistentHashRing(vnodes)`:
1. `add_node(node)` places `vnodes` **virtual points** on an md5-hash ring (kept sorted for bisect); `get_node(key)` walks clockwise to the owning vnode; `remove_node(node)` removes only that node's points.
2. **Prove the property empirically**: with 2,000 keys, adding a 4th node must reassign only ~1/(N+1) ≈ 25% of keys — while naive modulo reassigns ~75%. Print both numbers.
3. Verify the post-add distribution is balanced (~500 keys per node) and explain what virtual nodes do for balance on heterogeneous clusters.

**Follow-ups:** why md5 (or xxHash) over Python's built-in `hash()`? Where do replicas and weighted nodes fit? What gets worse as vnodes → ∞ (memory, add-node cost)?

In [0]:
# Solution Q4 — consistent hashing: add-node moves ~1/(N+1), modulo moves ~N/(N+1)
import hashlib, bisect
from collections import Counter

class ConsistentHashRing:
    def __init__(self, vnodes=150):
        self.vnodes = vnodes
        self.ring = []          # sorted vnode hash positions
        self.node_at = {}       # vnode hash -> physical node
    @staticmethod
    def _h(key):
        return int(hashlib.md5(key.encode()).hexdigest(), 16)  # stable across runs/processes
    def add_node(self, node):
        for v in range(self.vnodes):
            hv = self._h(f"{node}#v{v}")     # virtual points spread each node around the ring
            bisect.insort(self.ring, hv)
            self.node_at[hv] = node
    def remove_node(self, node):
        self.ring = [hv for hv in self.ring if self.node_at[hv] != node]
        self.node_at = {hv: n for hv, n in self.node_at.items() if n != node}
    def get_node(self, key):
        if not self.ring:
            raise KeyError("empty ring")
        hv = self._h(key)
        idx = bisect.bisect_right(self.ring, hv) % len(self.ring)   # first point clockwise
        return self.node_at[self.ring[idx]]

keys = [f"obj:{i}" for i in range(2000)]
ring = ConsistentHashRing(vnodes=150)
for node in ("cache-a", "cache-b", "cache-c"):
    ring.add_node(node)
before = {k: ring.get_node(k) for k in keys}

ring.add_node("cache-d")                        # the operational event: scale out
after = {k: ring.get_node(k) for k in keys}
moved_consistent = sum(1 for k in keys if before[k] != after[k])

h = ConsistentHashRing._h
moved_naive = sum(1 for k in keys if h(k) % 3 != h(k) % 4)     # modulo remap 3 -> 4

print(f"consistent hashing moved {moved_consistent/len(keys):.1%} of keys")
print(f"naive modulo moved    {moved_naive/len(keys):.1%} of keys")
assert 0.15 < moved_consistent / len(keys) < 0.35    # ~1/(N+1) = 25%
assert moved_consistent < moved_naive                 # ~75% for modulo

dist = Counter(after.values())
print("post-add distribution:", dict(dist))
assert len(dist) == 4 and all(350 <= v <= 700 for v in dist.values())   # balanced ~500 each

ring.remove_node("cache-b")                        # scale-in: only cache-b's keys move
assert all(ring.get_node(k) in ("cache-a", "cache-c", "cache-d") for k in keys[:100])
print("Q4 OK")
print("""
[SAY THIS OUT LOUD]
- Python's hash() is randomized per process (PYTHONHASHSEED) — md5/xxHash is the stable choice.
- Virtual nodes fix BOTH balance and heterogeneity: give a 2x node 2x vnodes; ~150 vnodes smooths
  the 1/(N+1) expectation to within a few percent.
- Hot keys: replicate each key R times by walking clockwise to R DISTINCT owners. Consistent
  hashing is the backbone of Dynamo, Cassandra token rings, and Kafka partition assignment.
""")

## Q5 — PySpark Coding: Arbitrary Stateful Streaming — Sessionization with `flatMapGroupsWithState` [ADVANCED]

**Interviewer (Uber):** Riders send pings while on-trip. A **trip session** ends when a rider goes silent for more than **10 minutes** (or a new gap appears). When a session completes, emit `(rider_id, session_id, n_pings, start, end)`. The pipeline runs 24/7.

1. Why don't `dropDuplicates` or plain windows suffice? When is `flatMapGroupsWithState` the ONLY tool that fits (custom timeout logic, partial emissions, custom state schema)?
2. Implement the state function: per-rider state holds `(session_id, n_pings, last_ts)`; on a gap > 10 min emit the OLD session and start a new one; use `GroupStateTimeout` as the backstop for riders who go silent forever.
3. Where does the state physically live (RocksDB/HDFS-backed state store)? What does 10M active keys cost, and how do state TTL and the watermark bound it?

In [0]:
# Solution Q5 — sessionization state machine + arbitrary-stateful streaming wiring
from pyspark.sql import functions as F, types as T
import uuid

GAP_MINUTES = 10

events = spark.createDataFrame([
    ("R1", "2026-11-01 08:00:00"),
    ("R1", "2026-11-01 08:05:00"),
    ("R1", "2026-11-01 08:40:00"),   # gap 35 min -> session 1 CLOSES here
    ("R1", "2026-11-01 08:44:00"),
    ("R2", "2026-11-01 09:00:00"),
    ("R2", "2026-11-01 09:11:00"),   # gap 11 min -> two sessions
    ("R3", "2026-11-01 10:02:00"),
    ("R3", "2026-11-01 10:04:00"),   # one session
    ("R4", "2026-11-01 11:00:00"),
    ("R4", "2026-11-01 11:30:00"),   # three isolated pings + one pair -> 3 sessions
    ("R4", "2026-11-01 12:00:00"),
    ("R4", "2026-11-01 12:02:00"),
], ["rider_id", "event_ts"]).withColumn("event_ts", F.col("event_ts").cast("timestamp"))

# ---------- (1) the state machine — pure, unit-testable, framework-free ----------
def advance(session, ts):
    """Advance an open session by one event. Returns ('close' | 'continue', session)."""
    if session is None:
        return "continue", {"session_id": str(uuid.uuid4()), "n_pings": 1,
                             "start_ts": ts, "last_ts": ts}
    if (ts - session["last_ts"]).total_seconds() > GAP_MINUTES * 60:
        return "close", {"session_id": str(uuid.uuid4()), "n_pings": 1,
                         "start_ts": ts, "last_ts": ts}
    session["n_pings"] += 1
    session["last_ts"] = ts
    return "continue", session

def sessionize(rider_id, timestamps):
    """Batch equivalent of the streaming state function — same logic, no state store."""
    out, cur = [], None
    for ts in sorted(timestamps):
        if cur and (ts - cur["last_ts"]).total_seconds() > GAP_MINUTES * 60:
            out.append({**cur, "rider_id": rider_id})       # gap rule closes the old session
            cur = None
        if cur is None:
            cur = {"session_id": str(uuid.uuid4()), "n_pings": 0, "start_ts": ts}
        cur["n_pings"] += 1
        cur["last_ts"] = ts
    if cur:
        out.append({**cur, "rider_id": rider_id})             # flush the still-open session
    return out

# ---------- (2) prove the state machine on the deterministic dataset ----------
per_rider = {}
for r in events.orderBy("event_ts").collect():
    per_rider.setdefault(r["rider_id"], []).append(r["event_ts"])

counts, all_sessions = {}, []
for rid, tss in per_rider.items():
    sessions = sessionize(rid, tss)
    all_sessions += sessions
    counts[rid] = len(sessions)

assert counts == {"R1": 2, "R2": 2, "R3": 1, "R4": 3}, counts
r1 = [s for s in all_sessions if s["rider_id"] == "R1"]
assert sorted(s["n_pings"] for s in r1) == [2, 2]           # (08:00,08:05) and (08:40,08:44)
print("session counts per rider:", counts)

# ---------- (3) the streaming wiring — what the real job looks like ----------
wiring = '''
# State schema persisted per rider in the state store:
state_schema = T.StructType([
    T.StructField("session_id", T.StringType()),
    T.StructField("n_pings",    T.IntegerType()),
    T.StructField("start_ts",   T.TimestampType()),
    T.StructField("last_ts",    T.TimestampType()),
])

# Arbitrary-stateful pattern (flatMapGroupsWithState / applyInPandasWithState, API by runtime):
sessions = (events_stream
    .withWatermark("event_ts", "30 minutes")        # bounds state for riders who vanish
    .groupBy("rider_id")
    .flatMapGroupsWithState(out_schema, state_schema,
                            GroupStateTimeout.EventTimeTimeout)(update_fn))
# update_fn = advance() above: pull state, apply the gap rule, EMIT the session row when it
# closes, push state back. EventTimeTimeout lets the watermark close sessions for riders
# who never come back - the state store then drops them (bounded memory).

# Built-ins cover ~80% of sessionization BEFORE you reach for this (name them first!):
#   groupBy(F.session_window("event_ts", "10 minutes"))   # Spark 3.2+
# Reach for arbitrary stateful ONLY when you need: emit-on-close output rows, custom TTL,
# or session merging across batches that session_window cannot express.
'''
print(wiring)

print("""
[NOTES TO SAY OUT LOUD]
- State lives in the STATE STORE (checkpoint): default in-memory (HDFS-backed files);
  RocksDB-backed provider for large key counts - 400M riders won't fit in-memory state.
- Watermark + EventTimeTimeout = the GC for state: riders who stop pinging are evicted once
  their watermark passes; without it, every rider_id lives FOREVER -> unbounded state.
- Timeout firing EMITS the final row: that's how a 'silent' session still gets closed.
- Checkpoint & state schema changes are breaking (see Q11) - version the state schema with
  care; test upgrades on a fresh checkpoint with replayed events.
- Cost: one state read+write per event. Design the state row to stay tiny (ids + timestamps,
  not payloads).
""")
print("Q5 OK — gap-rule state machine verified (2,2,1,3 sessions) + streaming wiring")

## Q6 — PySpark Coding: Stream-to-Stream Join — Impressions × Clicks [ADVANCED]

**Interviewer (Meta):** Join ad **impressions** to **click** events on `(ad_id, user_id)`. Clicks can arrive up to **5 minutes after** the impression. Unmatched impressions must still be counted (CTR = clicks ÷ impressions).

1. Why can't this be a batch join? Why do **BOTH** sides need watermarks — what state would grow unbounded otherwise?
2. Write the stream-stream **inner join** with event-time range condition `click_ts BETWEEN imp_ts AND imp_ts + interval 5 minutes` and watermarks on both sides.
3. Switching to a **left outer** join: what does Spark do differently for cleanup (why must it wait beyond the outer-side watermark), and what does that cost in state-store bytes? How do you bound it?

In [0]:
# Solution Q6 — dual-watermark stream-stream join (deterministic file-source demo)
from pyspark.sql import functions as F
import os

root = "/Volumes/workspace/default/etl_test_volume/d19_q6"
dbutils.fs.rm(root, True)  # idempotent re-run
imp_dir, click_dir = os.path.join(root, "imps"), os.path.join(root, "clicks")

# Engineer a deterministic, FINITE dataset:
#  imp1 (10:00) clicked at 10:03      -> match
#  imp2 (10:00) clicked at 10:05      -> match (exactly AT the 5-min edge, inclusive)
#  imp3 (10:00) clicked at 10:09      -> NO match (late beyond window)
#  imp4 (10:20) never clicked         -> unmatched impression (inner: dropped; outer: emitted)
spark.createDataFrame([
    ("imp1", "adA", "u1", "2026-11-01 10:00:00"),
    ("imp2", "adA", "u2", "2026-11-01 10:00:00"),
    ("imp3", "adB", "u3", "2026-11-01 10:00:00"),
    ("imp4", "adC", "u4", "2026-11-01 10:20:00"),
], ["imp_id", "ad_id", "user_id", "imp_ts"]).withColumn("imp_ts", F.col("imp_ts").cast("timestamp")) \
 .repartition(1).write.mode("overwrite").parquet(imp_dir)
spark.createDataFrame([
    ("clk1", "adA", "u1", "2026-11-01 10:03:00"),
    ("clk2", "adA", "u2", "2026-11-01 10:05:00"),   # exactly at the 5-min edge
    ("clk3", "adB", "u3", "2026-11-01 10:09:00"),
], ["clk_id", "ad_id", "user_id", "click_ts"]).withColumn("click_ts", F.col("click_ts").cast("timestamp")) \
 .repartition(1).write.mode("overwrite").parquet(click_dir)

imps  = spark.readStream.schema(spark.read.parquet(imp_dir).schema).parquet(imp_dir)
clicks = spark.readStream.schema(spark.read.parquet(click_dir).schema).parquet(click_dir)

GAP = "5 minutes"
imp_s   = imps.withWatermark("imp_ts", GAP).alias("imp")
click_s = clicks.withWatermark("click_ts", GAP).alias("click")

# PySpark join signature is (other, on, how) - the condition goes in as `on`
joined = imp_s.join(
    click_s,
    on=(F.col("imp.ad_id") == F.col("click.ad_id"))
       & (F.col("imp.user_id") == F.col("click.user_id"))
       & (F.col("click.click_ts").between(F.col("imp.imp_ts"),
                                           F.col("imp.imp_ts") + F.expr(f"interval {GAP}"))),
    how="inner")

ckpt = os.path.join(root, "ck")
q = (joined.writeStream.format("memory").queryName("q6_joined")
     .option("checkpointLocation", ckpt)
     .trigger(availableNow=True).start())
q.awaitTermination()

res = spark.sql("SELECT imp_id, clk_id, imp_ts, click_ts FROM q6_joined ORDER BY imp_id")
res.show(truncate=False)
matched = {r["imp_id"] for r in res.collect()}
assert matched == {"imp1", "imp2"}, matched   # imp3 clicked too late; imp4 never clicked
print("Q6 OK — dual watermarks joined imp1 + imp2 (edge click); imp3 (too late) and imp4 (no click) unmatched")
dbutils.fs.rm(root, True)  # clean up demo artifacts

# --- outer join variant (emits unmatched impressions once the window truly closes) ---
outer_q = (imps.withWatermark("imp_ts", GAP)
    .join(clicks.withWatermark("click_ts", GAP),
          ["ad_id", "user_id"], "leftOuter"))
print("""
[OUTER-JOIN SEMANTICS TO SAY OUT LOUD]
- With leftOuter, Spark can't discard unmatched impression state as soon as the click-side
  watermark passes: it must keep each impression until imp_ts + 5min < BOTH watermarks,
  then emit it with NULLs. State lives LONGER than the inner join — that's the cost of CTR.
- Bounding it: watermarks on both sides (mandatory) + small enough grace window + RocksDB
  state store (spark.sql.streaming.stateStore.providerClass = rocksdb) for 100M+ keys.
""")

## Q7 — Spark Optimization: Spark Connect & the UDF Serialization Tax [ADVANCED]

**Interviewer (Databricks):** Spark Connect (the client/server architecture behind serverless compute) moved plan-building off the cluster. Explain the architecture, then **measure** what it means for Python UDFs on 1M rows.

1. **Architecture:** the DataFrame API serializes to a protobuf plan over gRPC; the server-side driver optimizes and executes; results stream back as Arrow batches. Why does this enable serverless compute pools, thin clients, and Spark version decoupling?
2. **The tax:** a row-at-a-time Python UDF pickles every row to a Python worker and back (IPC per row); a pandas UDF moves whole Arrow batches (thousands of rows per serialization); a **native expression never leaves the JVM** (and may hit Tungsten codegen / Photon).
3. **Demo:** implement `x² + 1` three ways — native, pandas UDF, row UDF — on 1M cached rows, print wall-clock times, and assert the row UDF is the slowest.
4. **Rules of thumb:** native > pandas_udf > udf; avoid UDFs closing over big objects; know when a pandas UDF BEATS everything (when it replaces a join+shuffle with local batching).

In [0]:
# Solution Q7 — architecture talk track + measured UDF tax on 1M rows
from pyspark.sql import functions as F
import time

df = (spark.range(1_000_000).withColumn("x", F.col("id") % 97 + 1).cache())
df.count()   # materialize the cache so we time SERIALIZATION, not data generation

# --- 1) native expression: stays in the JVM plan (codegen/Photon eligible) ---
t0 = time.time()
df.withColumn("y", F.col("x") * F.col("x") + 1).agg(F.sum("y")).collect()
t_native = time.time() - t0

# --- 2) pandas UDF: data crosses to Python once per BATCH (Arrow) ---
@F.pandas_udf("long")
def fast_sq(s):
    return s * s + 1
t0 = time.time()
df.withColumn("y", fast_sq("x")).agg(F.sum("y")).collect()
t_pandas = time.time() - t0

# --- 3) row-at-a-time UDF: Python pickling + IPC per ROW ---
row_udf = F.udf(lambda v: v * v + 1, "long")
t0 = time.time()
df.withColumn("y", row_udf("x")).agg(F.sum("y")).collect()
t_row = time.time() - t0

print(f"native expr {t_native:.2f}s | pandas UDF {t_pandas:.2f}s | row UDF {t_row:.2f}s")
assert t_row > 1.5 * t_native    # the row tax is real on this cluster too
print("Q7 OK")
print("""
[SAY THIS OUT LOUD — Spark Connect architecture]
- The client (this notebook) builds the logical plan; the API serializes it as protobuf over gRPC;
  the SERVER-side driver optimizes + executes; results stream back as Arrow batches. Serverless
  pools start in seconds and client/server Spark versions decouple because of this split.
- The tax: a row UDF ships EVERY ROW to a Python worker and back (pickle + IPC per row);
  a pandas UDF ships Arrow batches; a native expression never leaves the JVM.
- Rules of thumb: native > pandas_udf > udf. Never close over large objects (they pickle with the
  UDF). A pandas UDF can BEAT everything when it replaces a join+shuffle with local batching.
""")

## Q8 — Spark Optimization: Sort-Merge vs Shuffled Hash Join [HARD]

**Interviewer (Netflix):** After the shuffle, your join has a **2 GB build side** and a **4 TB probe side**, spread over 2,000 partitions on 8-core executors. Spark planned a **sort-merge join (SMJ)**. Your peer claims a **shuffled hash join (SHJ)** would be faster. Who's right, and why?

1. Compare the two mechanically: SMJ = sort **both** sides, merge-walk; SHJ = hash the build side into memory, stream-probe the other. Cost of each per side.
2. Why is SMJ the default if SHJ does less work? What is the OOM risk of SHJ, and how does it relate build-side size to **shuffle partition count**?
3. How do you actually get an SHJ: `spark.sql.join.preferSortMergeJoin=false`, the `/*+ SHUFFLE_HASH(t) */` hint, and the case where one side is pre-sorted anyway. State the sizing rule of thumb.

In [0]:
# Solution Q8 — join strategy by build-side size + the SHUFFLE_HASH hint
from pyspark.sql import functions as F
import io, contextlib

def join_plan(df):
    """Capture the physical plan (works on classic clusters AND Spark Connect)."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def join_lines(plan):
    return [l.strip() for l in plan.splitlines() if "Join" in l]

# --- regime 1: small build side -> the planner AUTO-broadcasts it (cheapest of all) ---
small_build = spark.range(0, 100_000).withColumn("k", F.col("id") % 1000)
probe       = spark.range(0, 2_000_000).withColumn("k", F.col("id") % 1000)   # 'large' probe side
plan_small = join_plan(small_build.join(probe, "k"))
print("SMALL BUILD ->", join_lines(plan_small)[:1])       # BroadcastHashJoin

# --- regime 2: too big to broadcast -> shuffled-join territory (explain is lazy: nothing runs) ---
# Generated-range stats are optimistic (Photon broadcast even 20M rows) and the threshold
# conf is locked on serverless - so we pin the strategies with per-query hints: MERGE forces
# exactly what classic Spark plans by default for a 2GB-build / 4TB-probe join
build = spark.range(0, 20_000_000).withColumn("k", F.col("id") % 1000)          # ~320MB estimated
plan_smj = join_plan(build.hint("MERGE").join(probe, "k"))
join_smj = join_lines(plan_smj)
print("MERGE-hint  ->", join_smj[:1])       # SortMergeJoin (classic Spark default when too big to broadcast)

# --- regime 3: force shuffled hash join via the per-query hint ---
plan_shj = join_plan(build.hint("SHUFFLE_HASH").join(probe, "k"))
join_shj = join_lines(plan_shj)
print("HINTED      ->", join_shj[:1])

assert "Broadcast" in join_lines(plan_small)[0], join_lines(plan_small)[:3]
assert "SortMergeJoin" in plan_smj, join_smj[:3]
assert "ShuffledHashJoin" in plan_shj, join_shj[:3]
print("\nQ8 OK — broadcast when small, shuffled join when big, SHUFFLE_HASH hint forced SHJ")

print("""
[MECHANICS]
SMJ: sort(build) + sort(probe) + merge walk. Work: O(B log B) + O(P log P) + one scan each.
SHJ: hash build side into per-partition hash tables (RAM), then STREAM the probe side. No sorts.

[WHY CLASSIC SPARK DEFAULTS TO SMJ (and modern runtimes sometimes don't)]
SHJ must hold the ENTIRE build-side partition in memory per task. If build_per_partition >
execution memory -> OOM (the hash table cannot spill in classic SHJ). SMJ is spill-safe:
both phases spill to disk. Classic Spark refuses to gamble OOM on a plan that only MIGHT
fit - so it plans SMJ and lets you opt into SHJ. Modern engines (Photon / AQE) know real
shuffle sizes at runtime and will pick SHJ THEMSELVES when it is provably safe - exactly
the sizing logic below, which is why you should be able to recite it.

[THE SIZING RULE]
build_per_partition ~= build_bytes / shuffle_partitions. With 2,000 partitions, a 2GB build
side = ~1MB/partition -> trivially fits. With 20 partitions -> 100MB/partition -> still OK
on a 16g heap, but at 2GB/partition you are back to OOM risk. SHJ wins when: partitions are
plentiful, the probe side is huge (sorting 4TB is expensive!), build per-partition is small.

[HOW TO FORCE IT]
1. spark.sql.join.preferSortMergeJoin=false     (cluster-wide, affects ALL joins - risky)
2. df.hint('SHUFFLE_HASH')                      (per-query, preferred)
3. spark.sql.autoBroadcastJoinThreshold=-1      (forbids broadcast; classic clusters - locked on serverless)
4. If a side is ALREADY sorted (e.g. bucketed on the join key) SMJ skips the sort -> keep SMJ.
5. AQE can also rewrite plans at runtime once real shuffle sizes are known.

[ANSWER TO THE INTERVIEWER] With 2,000 partitions and a 2GB build side, the peer is right:
SHJ skips sorting the 4TB probe side and ~1MB build partitions fit safely in memory. Force it
per-query with the hint, then confirm zero spill in the Spark UI.
""")

## Q9 — ETL: Zero-Downtime Schema Migration — Expand & Contract [ADVANCED]

**Interviewer (Stripe):** Split `full_name` into `first_name` / `last_name` on a gold table consumed by **15 dashboards and 3 downstream pipelines**. Zero downtime, zero breakage, rollback-safe.

1. Explain **expand-contract**: expand (add new columns, dual-write), migrate (backfill history), contract (drop old column). Why can't you just `ALTER TABLE ... RENAME`?
2. How does a **blue-green view swap** make the cutover atomic for consumers? Show the sequence.
3. Why must the **contract phase wait** for consumer verification? How do you detect consumers still reading `full_name` (query history / access logs) before dropping it?

In [0]:
# Solution Q9 — expand-contract with blue-green view swap (runnable on a volume-backed path)
from pyspark.sql import functions as F

tbl = "/Volumes/workspace/default/etl_test_volume/d19_q9/customers"
dbutils.fs.rm(tbl, True)  # idempotent re-run

# --- original table (old world) ---
spark.createDataFrame([(1, "Grace Hopper", "COBOL Corp", "2026-10-01"),
                       (2, "Alan Turing", "Enigma Ltd", "2026-10-05")],
                      ["id", "full_name", "company", "signup"]) \
     .write.format("delta").save(tbl)

# --- PHASE 1: EXPAND (non-breaking: additive only) ---
spark.sql(f"ALTER TABLE delta.`{tbl}` ADD COLUMNS (first_name STRING, last_name STRING)")

# --- PHASE 2: DUAL-WRITE (writer populates BOTH old and new columns) ---
new_rows = spark.createDataFrame([(3, "Ada Lovelace", "Analytical Eng", "2026-10-28")],
                                 ["id", "full_name", "company", "signup"])
new_rows = (new_rows
    .withColumn("first_name", F.split("full_name", " ").getItem(0))
    .withColumn("last_name", F.element_at(F.split("full_name", " "), -1)))
new_rows.write.format("delta").mode("append").save(tbl)

# --- PHASE 3: MIGRATE / BACKFILL history ---
spark.read.format("delta").load(tbl).createOrReplaceTempView("cust_v")
spark.sql(f"""
MERGE INTO delta.`{tbl}` t
USING (
  SELECT id,
         split(full_name, ' ')[0]               AS first_name,
         element_at(split(full_name, ' '), -1)  AS last_name
  FROM cust_v WHERE first_name IS NULL
) s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.first_name = s.first_name, t.last_name = s.last_name
""")

# --- PHASE 4: BLUE-GREEN VIEW SWAP (atomic for every consumer) ---
# Consumers read the VIEW, never the base table. The flip is one CREATE OR REPLACE -> atomic.
spark.sql(f"""CREATE OR REPLACE VIEW customers_v2 AS
             SELECT id, first_name, last_name, company, signup, full_name
             FROM delta.`{tbl}`""")

# --- PHASE 5: CONTRACT (only after query logs show nobody reads full_name) ---
chk = spark.sql(f"SELECT id, first_name, last_name FROM delta.`{tbl}` ORDER BY id")
chk.show()
assert chk.filter("first_name IS NULL OR last_name IS NULL").count() == 0  # backfill complete
assert chk.count() == 3

print("""
[CONTRACT CHECKLIST]
- Scan query history / UC audit logs for 'full_name' reads over the last N days; empty -> safe
  to DROP COLUMN. Non-empty -> keep deriving full_name in the view for stragglers.
- Rollback at any phase is trivial: additive columns are ignored by old readers until the swap;
  the view flip is reversible in one statement. ALTER ... RENAME would have broken every
  consumer INSTANTLY and irreversibly - exactly what expand-contract avoids.
""")
print("Q9 OK — expand-contract executed end-to-end with atomic view swap")
spark.sql("DROP VIEW IF EXISTS customers_v2")
dbutils.fs.rm(tbl, True)  # clean up demo artifacts

## Q10 — ETL: DEEP CLONE — Backup, DR & Environment Seeding [ADVANCED]

**Interviewer (Adobe):** Before a risky reprocessing job, you must copy a production gold table into a staging environment. The table is 5 TB with 60 days of history. Compare your options and write the workflow.

1. **Deep clone vs shallow clone**: what is physically copied (data files vs metadata)? What does a shallow clone cost, and when is it a trap (source deleted/vacuumed)?
2. Write the workflow: clone -> run the risky job on the clone -> validate parity -> **refresh** the clone from an updated source. What is the **clone-on-write** behavior when the source keeps changing?
3. Position deep clone against `RESTORE`/time travel and `VACUUM` retention in an overall DR strategy — what protects against what?

In [0]:
# Solution Q10 — shallow vs deep clone, isolation check, incremental refresh (managed tables)
SRC, DEEP, SHALLOW = ("workspace.default.d19_q10_prod",
                      "workspace.default.d19_q10_deep",
                      "workspace.default.d19_q10_shallow")
for t in (SRC, DEEP, SHALLOW):
    spark.sql(f"DROP TABLE IF EXISTS {t}")   # idempotent re-run

# --- the production gold table (managed) ---
spark.sql(f"""CREATE OR REPLACE TABLE {SRC} AS
             SELECT * FROM VALUES
               (1, '2026-10-01', 120.0),
               (2, '2026-10-02',  80.0) AS t(order_id, dt, amount)""")

# --- SHALLOW clone: metadata only; data files are SHARED with the source ---
spark.sql(f"CREATE OR REPLACE TABLE {SHALLOW} SHALLOW CLONE {SRC}")

# --- DEEP clone: physical copy of data files + full version history ---
spark.sql(f"CREATE OR REPLACE TABLE {DEEP} DEEP CLONE {SRC}")

# --- run the 'risky job' on the CLONE, not on prod ---
spark.sql(f"INSERT INTO {DEEP} VALUES (3, '2026-10-03', 42.0)")

# --- parity validation: prod untouched, clone diverged (isolation worked) ---
prod_rows    = spark.sql(f"SELECT count(*) FROM {SRC}").first()[0]
deep_rows    = spark.sql(f"SELECT count(*) FROM {DEEP}").first()[0]
shallow_rows = spark.sql(f"SELECT count(*) FROM {SHALLOW}").first()[0]
assert (prod_rows, deep_rows, shallow_rows) == (2, 3, 2), (prod_rows, deep_rows, shallow_rows)
print(f"prod={prod_rows}, deep_clone={deep_rows}, shallow_clone={shallow_rows} -> risky job fully isolated")

# --- source keeps changing; incremental refresh: re-issue the same DEEP CLONE ---
spark.sql(f"INSERT INTO {SRC} VALUES (4, '2026-10-04', 9.0)")
spark.sql(f"CREATE OR REPLACE TABLE {DEEP} DEEP CLONE {SRC}")
assert spark.sql(f"SELECT count(*) FROM {DEEP}").first()[0] == 3  # refreshed to source state

print("""
[WHAT TO SAY OUT LOUD]
- SHALLOW clone copies metadata/transaction log only; files SHARED with the source. Instant,
  free-ish. TRAP: if the source is vacuumed or dropped, the shallow clone's old versions
  break -> never a long-lived backup. Fine for dev/testing at prod scale.
- DEEP clone copies data files + full version history. Cost = storage. Survives source VACUUM.
- On Unity Catalog, clones are MANAGED tables (the old path-based LOCATION form needs a
  cloud-scheme URI and is legacy; shallow clone in particular only supports managed tables).
- Clone-on-write: writes to either side create NEW files; the shared prefix is never mutated.
- Refresh: re-issuing the same DEEP CLONE copies only files added since the last clone.
- DR matrix: time travel/RESTORE -> logical mistakes on the SAME table (bounded by VACUUM
  retention, default 7 days); DEEP CLONE -> independent copy immune to source vacuum/drop;
  cross-region replication -> site failure. Time travel is the oops layer, clone is the DR layer.
""")
for t in (SHALLOW, DEEP, SRC):
    spark.sql(f"DROP TABLE IF EXISTS {t}")   # clean up demo artifacts
print("Q10 OK — shallow vs deep cloned, risky job isolated, clone refreshed incrementally")

## Q11 — ETL: Zero-Loss Streaming Upgrades — Checkpoints & State [HARD]

**Interviewer (Uber):** A 24/7 Kafka→Delta pipeline (deduplication + running aggregates, stateful, 400M keys) must move to a new Spark version **with a changed aggregation**. No event may be lost or double-counted. Walk me through the upgrade.

1. What exactly lives in a **checkpoint** (offsets, commits, state store)? Why does changing the query shape break it — what error would you hit?

2. Give the **safe upgrade runbook**: drain the query with `Trigger.AvailableNow`, validate final state, start the new version on a **fresh checkpoint**, replay the overlap window, verify parity with counts/checksums. Which state-store configs matter (RocksDB provider, changelog checkpointing)?

3. Classify changes: **checkpoint-compatible** (adding a column) vs **incompatible** (changing groupBy keys / aggregation logic / watermark column). How do you migrate state when the change is incompatible?

In [0]:
# Solution Q11 — safe-upgrade runbook (playbook + the drain pattern in code)

RUNBOOK = {
  "1. DRAIN": """Set Trigger.AvailableNow (processes ALL pending source data as finite batches,
      then stops cleanly). awaitTermination. Offsets fully committed -> nothing in flight.
      This is the ONLY clean way to stop a stateful query - never kill -9 it.""",
  "2. FREEZE A SNAPSHOT": """Backup the Delta sink + the checkpoint dir (offsets/, commits/,
      state/) + record the final watermark and last committed offset per Kafka partition.""",
  "3. PREPARE THE NEW QUERY": """New Spark version, new aggregation. Point it at a FRESH
      checkpoint location - NEVER reuse the old checkpoint across incompatible query changes.""",
  "4. RESEED THE OVERLAP": """Replay from the last committed offset (set startingOffsets for
      Kafka; idempotent reprocess of the window for file sources). The new aggregation
      rebuilds state from scratch over the overlap window.""",
  "5. DEDUPE THE SINK": """Writes must be idempotent: MERGE on (partition, offset) keys or
      REPLACEWHERE the overlap window so replayed events do NOT double-write.""",
  "6. VERIFY PARITY BEFORE SWITCH": """counts, sums, row-level checksums old vs new across the
      overlap; only then repoint consumers. Keep the old pipeline paused for instant rollback.""",
}
for step, text in RUNBOOK.items():
    print(step, "->", text.replace("\n      ", " "), "\n")

# --- the drain pattern (the code they want to SEE) ---
drain_snippet = '''
# OLD pipeline: switch trigger to AvailableNow to drain instead of killing it
q = (streaming_df.writeStream
       .format("delta")
       .option("checkpointLocation", ckpt)
       .trigger(availableNow=True)     # <-- finite: drains, commits, stops
       .start())
q.awaitTermination()                  # clean stop; offsets committed

# NEW pipeline: fresh checkpoint + RocksDB state store + changelog checkpointing
spark.conf.set("spark.sql.streaming.stateStore.providerClass",
               "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")
spark.conf.set("spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled",
               "true")
q2 = (new_streaming_df.writeStream
        .format("delta")
        .option("checkpointLocation", NEW_CKPT)  # fresh - never reuse across incompatible changes
        .trigger(processingTime="30 seconds")
        .start())
'''
print(drain_snippet)

COMPAT = {
  "checkpoint-COMPATIBLE (may reuse checkpoint)": [
    "adding a new output column / sink field",
    "changing trigger cadence",
    "map-only output changes that don't touch state keys",
  ],
  "INCOMPATIBLE (fresh checkpoint + reseed required)": [
    "changing groupBy keys / flatMapGroupsWithState key or state schema",
    "changing aggregation logic",
    "changing the watermark column or delay",
    "Spark version jumps that change state-store format versions",
  ],
  "migrating state when incompatible": [
    """export state to a Delta table (foreachBatch flush), transform it, then load it as the
     initial seed for the new query - rebuild state, never port it""",
  ],
}
for k, v in COMPAT.items():
    print(k + ":")
    for x in v:
        print("   -", x)
print("\nQ11 OK — drain pattern + fresh-checkpoint rule + compatibility matrix")

## Q12 — Data Modeling: Event Sourcing vs State Tables [ADVANCED]

**Interviewer (Uber):** Checkout stores only the CURRENT order state (a mutable table). Finance now needs as-of answers, a full audit trail, and the ability to **restate history** when business rules change. The state table can't do any of it. Redesign.

1. Model the alternative: `order_events` is an append-only, immutable event log (created / paid / shipped / returned) — the source of truth.
2. Derive the current-state projection with **conditional aggregation** (MAX(CASE ...) per event type) — rebuildable from the log at any time.
3. **Restate without touching raw events:** a rule change — "returned" now triggers a full refund — becomes a NEW projection (`net_revenue`), not a data fix. Show gross vs net side by side from the same event log.

**Follow-ups:** when is event sourcing overkill (plain CRUD, no audit need)? How do you version event schemas without breaking old consumers (upcasting)? What bounds replay cost (snapshots)? How does this differ from bitemporal "as-known-at" modeling?

In [0]:
# Solution Q12 — immutable event log + derived projections + restatement without edits
spark.createDataFrame([
    ("o1", "1", "created",  "2026-01-01",   0.0),
    ("o1", "1", "paid",     "2026-01-02",  89.0),
    ("o1", "1", "shipped",  "2026-01-04",   0.0),
    ("o2", "2", "created",  "2026-01-03",   0.0),
    ("o2", "2", "paid",     "2026-01-04", 120.0),
    ("o3", "3", "created",  "2026-01-02",   0.0),
    ("o3", "3", "paid",     "2026-01-03",  45.0),
    ("o3", "3", "shipped",  "2026-01-05",   0.0),
    ("o3", "3", "returned", "2026-01-09",   0.0),
], "order_id string, customer_id string, event_type string, event_time string, amount double").createOrReplaceTempView("q12_events")

projection = spark.sql("""
SELECT order_id,
  MAX(CASE WHEN event_type = 'created'  THEN event_time END)   AS created_at,
  MAX(CASE WHEN event_type = 'paid'     THEN amount END)       AS paid_amount,
  MAX(CASE WHEN event_type = 'shipped'  THEN event_time END)  AS shipped_at,
  MAX(CASE WHEN event_type = 'returned' THEN 1 ELSE 0 END)    AS is_returned,
  COUNT(*)                                                    AS event_count
FROM q12_events
GROUP BY order_id
ORDER BY order_id""")
projection.show()

p = {r["order_id"]: r for r in projection.collect()}
assert p["o1"]["paid_amount"] == 89.0 and p["o1"]["is_returned"] == 0
assert p["o3"]["is_returned"] == 1 and p["o3"]["paid_amount"] == 45.0
assert p["o2"]["event_count"] == 2

# --- restatement: NEW business rule, NEW projection — raw events untouched ---
revised = spark.sql("""
SELECT order_id,
  MAX(CASE WHEN event_type = 'paid' THEN amount END) AS gross_paid,
  CASE WHEN MAX(CASE WHEN event_type = 'returned' THEN 1 ELSE 0 END) = 1 THEN 0.0
       ELSE MAX(CASE WHEN event_type = 'paid' THEN amount END) END AS net_revenue
FROM q12_events
GROUP BY order_id
ORDER BY order_id""")
revised.show()
rv = {r["order_id"]: r for r in revised.collect()}
assert rv["o3"]["net_revenue"] == 0.0 and rv["o1"]["net_revenue"] == 89.0
print("Q12 OK — finance restated history by rebuilding a projection, not by editing data")
print("""
[SAY THIS OUT LOUD]
- Event store = source of truth; every "table" the business sees is a REPLAYABLE projection.
- Schema versioning: add event_version; consumers upcast old events (v1 -> v2 with defaults)
  instead of rewriting history.
- Snapshots (state at time T) bound replay cost: replay from the last snapshot, not from zero.
- Overkill when: no audit / as-of / restatement needs — a mutable state table is simpler.
""")

## Q13 — Data Modeling: Feature Store — Point-in-Time Correctness [ADVANCED]

**Interviewer (Netflix):** Your churn model's offline training data joined each label to the LATEST feature row for that customer. In production the model collapsed — because at scoring time you never have future features. This is **label leakage**. Design the feature tables correctly.

1. **Schema:** feature table `(entity_key, feature_ts, features...)`, label table `(entity_key, label_ts, label)`.
2. **Point-in-time join:** for each label, take the LATEST feature row with `feature_ts < label_ts` — join on entity + strict inequality, `ROW_NUMBER` take 1.
3. **Prove the bug:** show the naive "latest features" join handing c1's 2026-01-05 label the 2026-01-06 feature row (visits 30 instead of 12) — then show the PIT join correctly returning 12.

**Follow-ups:** online/offline parity (the same feature transformation must serve training AND low-latency scoring); freshness tiers; why daily aggregation spines make PIT joins cheap at scale; and how UC feature tables + entity models formalize this pattern.

In [0]:
# Solution Q13 — point-in-time join, with the naive leak demonstrated for contrast
spark.createDataFrame([
    ("c1", "2026-01-03", 12, 80.5),
    ("c1", "2026-01-06", 30, 210.0),   # AFTER the label date 01-05 -> must never leak
    ("c2", "2026-01-07",  5, 12.0),
    ("c2", "2026-01-10",  9, 30.0),    # AFTER the label date 01-08 -> must never leak
], "entity string, feature_date string, visits int, spend double").createOrReplaceTempView("q13_features")
spark.createDataFrame([
    ("c1", "2026-01-05", 1),
    ("c2", "2026-01-08", 0),
], "entity string, label_date string, churned int").createOrReplaceTempView("q13_labels")

pit = spark.sql("""
WITH joined AS (
  SELECT l.entity, l.label_date, l.churned, f.feature_date, f.visits, f.spend,
         ROW_NUMBER() OVER (PARTITION BY l.entity, l.label_date
                            ORDER BY f.feature_date DESC) AS rn
  FROM q13_labels l
  JOIN q13_features f
    ON  l.entity = f.entity
    AND f.feature_date < l.label_date            -- STRICT: only the PAST is knowable
)
SELECT entity, label_date, churned, feature_date, visits, spend
FROM joined WHERE rn = 1
ORDER BY entity""")
pit.show()

p = {r["entity"]: r for r in pit.collect()}
assert p["c1"]["feature_date"] == "2026-01-03" and p["c1"]["visits"] == 12
assert p["c2"]["feature_date"] == "2026-01-07" and p["c2"]["visits"] == 5

# --- the naive "latest features" join, for contrast: it hands the model the future ---
leak = spark.sql("""
SELECT f.entity, f.visits
FROM (SELECT entity, visits,
             ROW_NUMBER() OVER (PARTITION BY entity ORDER BY feature_date DESC) AS rn
      FROM q13_features) f
JOIN q13_labels l ON f.entity = l.entity
WHERE f.rn = 1""")
leak_v = {r["entity"]: r["visits"] for r in leak.collect()}
assert leak_v["c1"] == 30
print(f"Q13 OK — naive join leaks visits={leak_v['c1']} (from 01-06); PIT join correctly uses {p['c1']['visits']} (from 01-03)")
print("""
[SAY THIS OUT LOUD]
- Leakage = training on information unavailable at scoring time; the model looks great offline
  and collapses in prod. The PIT join makes offline match online.
- Scale: entity + feature_ts < label_ts can fan out; pre-aggregate features onto a DAILY SPINE
  (one row per entity per day) so the as-of lookup stays a cheap range join.
- Parity: the same feature table must feed offline training and online serving (or an online
  store mirroring it) — feature REUSE across models is the whole point of a feature store.
""")

## Q14 — Snowflake: GenAI Pipelines with Cortex AI Functions [ADVANCED]

**Interviewer (Adobe):** A Snowflake shop wants LLM enrichment INSIDE data pipelines — summarize support tickets, classify sentiment, and embed docs for RAG — without shipping data to an external API. Walk the 2025-2026 native stack and the judgment calls.

1. **Functions:** `SNOWFLAKE.CORTEX.COMPLETE` / `EMBED_TEXT` / `SENTIMENT` / `CLASSIFY_TEXT` invoked directly in SQL; unstructured data (PDFs, docs) via directory tables + semantic `SEARCH`; what compute they run on (serverless inference, per-token cost).
2. **Governance:** data never leaves the account boundary — RBAC, masking, and row-access policies apply to the model OUTPUT too. Why is that the argument vs BYO external API keys?
3. **When NOT native:** fine-tuning, long agentic workflows, or data already outside Snowflake — external models via external access integrations, and the hybrid pattern that keeps PII on-platform.
4. **Pipeline engineering:** idempotency (cache completions keyed by input hash), cost control (small model first, escalate on low confidence), and eval gates before a GenAI-enriched column goes gold.

*(No Snowflake connection in this workspace — the solution is a print-based talk track with representative SQL.)*

In [0]:
# Solution Q14 — Cortex talk track + representative SQL (print-based: no Snowflake here)
sql_examples = [
    ("SUMMARIZE tickets in the pipeline",
     "CREATE OR REPLACE TABLE support.ticket_summaries AS "
     "SELECT ticket_id, "
     "  SNOWFLAKE.CORTEX.COMPLETE('summarize this ticket in two sentences', body) AS summary "
     "FROM support.tickets;"),
    ("CLASSIFY + sentiment in one pass",
     "SELECT ticket_id, "
     "  SNOWFLAKE.CORTEX.CLASSIFY_TEXT(body, ['billing', 'bug', 'feature']) AS category, "
     "  SNOWFLAKE.CORTEX.SENTIMENT(body) AS sentiment "
     "FROM support.tickets;"),
    ("EMBED for RAG (semantic search index)",
     "CREATE OR REPLACE TABLE support.doc_embeddings AS "
     "SELECT doc_id, chunk, SNOWFLAKE.CORTEX.EMBED_TEXT('snowflake-arctic-embed', chunk) AS embedding "
     "FROM support.doc_chunks;"),
    ("Semantic SEARCH over unstructured files",
     "SELECT relative_path, SNOWFLAKE.CORTEX.SEARCH('refund policy', @docs_stage) "
     "FROM DIRECTORY(@docs_stage);"),
]
for label, sql in sql_examples:
    print(f"-- {label}")
    print(sql, end="\n\n")

TALK_TRACK = [
    ("cost model",
     "Per-token pricing on serverless inference — no warehouse to size. Control it: smallest model "
     "first (COMPLETE on a mini model), escalate only on low confidence; cache completions keyed by "
     "md5(input) so pipeline re-runs cost zero."),
    ("idempotency",
     "GenAI columns are DERIVED data: store input_hash + output and MERGE ON input_hash, so only "
     "NEW or CHANGED rows call the model. Re-running enrichment must not re-bill."),
    ("governance",
     "Data stays inside the account boundary: RBAC, masking and row-access policies apply to the "
     "model OUTPUT too. That is the argument vs BYO external API keys, which egress PII."),
    ("eval gates",
     "A GenAI-enriched column needs quality checks before gold: sampled human review scores, "
     "refusal/hallucination rate, and a regression suite against a golden set — same discipline as "
     "any pipeline column."),
    ("when NOT native",
     "Fine-tuning, long agentic workflows, or data already outside Snowflake -> external models via "
     "external access integrations; keep the PII-bearing steps on-platform (hybrid)."),
]
for k, v in TALK_TRACK:
    print(f"[{k.upper()}]")
    print(f"  {v}")
print()
print("Q14 OK — talk track delivered; representative SQL above is syntax-faithful")

## Q15 — Snowflake: Event Tables & Snowflake Logging [ADVANCED]

**Interviewer (Netflix):** A Snowpark stored procedure fails intermittently for **one tenant only**. You need logs and exceptions from inside Snowflake code — `print` goes nowhere.

1. What is an **event table**? How does it differ from a normal table (append-only, structured logging schema, lifecycle-managed retention)?
2. Use **Snowflake Logging** inside a Snowpark UDF/procedure: `logging.info/error`, the `TRACE_ON` level, and `EXCEPTION` events. Where do records land — name the key columns you query (`TIMESTAMP`, `RECORD`, `EVENT_NAME`, `SEVERITY`)?
3. Cost and control: how do log levels throttle volume, what's the retention lifecycle, and why is an event table better than a logging table you hand-roll?

In [0]:
# Solution Q15 — event table setup + Snowpark logging + the query you debug with

setup_sql = """
-- one event table per account (convention: <org>_events)
CREATE EVENT TABLE IF NOT EXISTS my_account_events;
-- route this user's logs/exceptions/traces to it (account or user level)
ALTER USER ANALYST_JANE SET EVENT_TABLE = mydb.events.my_account_events;

-- throttle what gets captured (default INFO; TRACE is a firehose - use sparingly)
ALTER SESSION SET LOG_LEVEL = INFO;
ALTER SESSION SET TRACE_LEVEL = ON_EVENT;   -- TRACE_ON(...) lines emit only when set
"""
print(setup_sql)

snowpark_code = '''
-- inside a Snowpark Python stored procedure
CREATE OR REPLACE PROCEDURE mydb.ops.process_tenant(tenant STRING)
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  PACKAGES = ('snowflake-snowpark-python')
  HANDLER = 'run'
AS $$
import logging

def run(session, tenant):
    log = logging.getLogger("tenant_proc")
    log.info("process_tenant START", extra={"tenant": tenant})
    try:
        rows = session.sql("SELECT COUNT(*) FROM facts WHERE tenant_id = ?", params=[tenant]).collect()
        n = rows[0][0]
        log.debug("row count fetched", extra={"tenant": tenant, "rows": n})
        log.info("process_tenant DONE", extra={"tenant": tenant, "rows": n})
        return str(n)
    except Exception:
        log.error("process_tenant FAILED", extra={"tenant": tenant})
        raise          # EXCEPT events also land in the event table automatically
$$;
'''
print(snowpark_code)

query_sql = """
-- the 3 AM query: this tenant's last failures with full detail
SELECT TIMESTAMP, EVENT_NAME, SEVERITY, RECORD, RESOURCE_ATTRIBUTES, EXCEPTION
FROM my_account_events
WHERE TIMESTAMP > DATEADD(hour, -24, CURRENT_TIMESTAMP())
  AND RESOURCE_ATTRIBUTES["snow.database.name"]:"string" = 'MYDB'
  AND RECORD["tenant"]:"string" = 'ACME'
  AND SEVERITY >= 'ERROR'
ORDER BY TIMESTAMP DESC;
"""
print(query_sql)

NOTES = {
  "columns you must be able to name":
    """TIMESTAMP, EVENT_NAME (LOG | EXCEPT | TRACE | SPAN), SEVERITY, RECORD (the structured
     key/value payload from extra={...}), RESOURCE_ATTRIBUTES (db/schema/user/object
     context), EXCEPTION (stack trace for EXCEPT events), RECORD_ATTRIBUTES""",
  "lifecycle/retention":
    """The event table is backed by lifecycle-managed storage: Snowflake moves old partitions
    to cheaper storage and expires them automatically. It is APPEND-ONLY - you cannot
    DELETE from it; control volume with log levels instead.""",
  "log levels as cost control":
    """ERROR / WARN / INFO / DEBUG plus TRACE. DEBUG and TRACE in steady state will drown the
    table and the bill - enable them per-session or with TRACE_LEVEL = ON_EVENT.""",
  "vs hand-rolled logging table":
    """Native: automatic context (user, object, role), zero ETL, lifecycle retention, and it
    works inside UDFs/procs where you have no file system. Hand-rolled: full control, but
    you rebuild all of that - and it usually can't log from inside UDF execution anyway.""",
}
for k, v in NOTES.items():
    print(f"[{k}]\n  {v}\n")
print("Q15 OK — event table + Snowpark logging + the debug query")

## Q16 — Monitoring & Orchestration: dbt on Databricks [ADVANCED]

**Interviewer (Spotify):** The SQL team lives in dbt; the Spark team lives in notebooks. How does dbt fit on Databricks, and who owns orchestration?

1. **Integration:** the dbt-databricks adapter; SQL warehouses + Photon (vs all-purpose clusters); materializations (`view` / `table` / `incremental` / `ephemeral`); dbt **tests as data-quality gates** and generated docs/lineage; Slim CI (`state:modified`).
2. **Orchestration decision matrix:** dbt Cloud scheduler vs Databricks Jobs (file-arrival / table-update triggers, Run If dependencies, repair runs) vs Airflow owning BOTH — and where each breaks (cross-tool lineage, backfills, environment promotion).
3. **Demo the core of `incremental` materialization:** a full-refresh build of a customer dim, then an incremental run where only changed + new rows MERGE on `unique_key` — untouched history is never rewritten.

**Follow-ups:** when is dbt the WRONG tool (PySpark-heavy pipelines → SDP / Lakeflow)? How do dbt exposures map to Unity Catalog governance? Who pages at 3 AM — the dbt job or the platform team?

In [0]:
# Solution Q16 — what dbt 'incremental + unique_key' really does, + the orchestration matrix
tgt = "workspace.default.d19_q16_customer_dim"
spark.sql("CREATE OR REPLACE TABLE " + tgt + " (customer_id INT, status STRING, updated_at STRING) USING DELTA")

# --- first run = dbt full refresh: the model builds the whole table ---
spark.sql("INSERT INTO " + tgt + " VALUES (1, 'active', '2026-01-01'), (2, 'active', '2026-01-01'), (3, 'churned', '2026-01-01')")
assert spark.table(tgt).count() == 3

# --- second run = dbt incremental: only the delta MERGEs in on unique_key ---
# (dbt compiles this MERGE for you; we spell it out because THIS is the interview answer)
spark.createDataFrame([(3, "active", "2026-02-01"), (4, "new", "2026-02-01")],
                      "customer_id INT, status STRING, updated_at STRING").createOrReplaceTempView("q16_incremental")
spark.sql("MERGE INTO " + tgt + " t USING q16_incremental s ON t.customer_id = s.customer_id "
          "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
rows = {r["customer_id"]: r["status"] for r in spark.table(tgt).collect()}
assert rows == {1: "active", 2: "active", 3: "active", 4: "new"}   # 3 corrected, 4 added
assert spark.table(tgt).count() == 4                                 # history NOT rewritten

ORCHESTRATION = {
    "dbt owns it": "Good when dbt Cloud already schedules + you want built-in docs/tests. Breaks on "
                   "Databricks-native triggers (file arrival, table update), repair runs, and anything "
                   "non-SQL.",
    "Jobs owns it": "dbt runs as a task inside a Lakeflow Job next to your PySpark/SDP tasks -> ONE "
                    "lineage, ONE retry/repair story, Run If dependencies, table-update triggers. The "
                    "usual recommendation on a Databricks-first platform.",
    "Airflow owns both": "Enterprise-wide scheduling, SLAs, cross-platform DAGs. Cost: you own "
                          "triggering, backfills, and the lineage glue across tools.",
    "tests as gates": "dbt test failures fail the task -> downstream table-update triggers never fire "
                      "-> bad data cannot cascade. That is the monitoring hook.",
}
for k, v in ORCHESTRATION.items():
    print(f"[{k}]")
    print(f"  {v}")
print()
print("Q16 OK — incremental MERGE verified: only changed + new rows written, history untouched")

## Q17 — Monitoring & Testing: Unit-Testing Data Pipelines (pytest + chispa) [ADVANCED]

**Interviewer (Spotify):** A currency-conversion transform silently rounds wrong for one locale, and it shipped to production. How do you add tests so it can never ship again?

1. What CAN be unit-tested in a data pipeline (pure transforms, UDFs, expectation logic) vs what needs integration tests (sources, sinks, cluster configs)? Sketch the **test pyramid** for data.
2. Write pytest cases for a PySpark transform using **chispa** (`assert_df_equality`, `assert_schema_equality`) — including the locale/rounding case that was broken.
3. Where do these tests run (CI pre-merge, `%pip install pytest chispa`), and what's the golden-dataset / snapshot strategy for regressions?

In [0]:
# Solution Q17 — pure transform + chispa/pytest test harness
# (in a real repo: %pip install pytest chispa --quiet, then `pytest tests/` in CI)
from pyspark.sql import functions as F, types as T
from decimal import Decimal

try:
    import chispa
    CHISPA = True
except ImportError:
    CHISPA = False
    print("[note] chispa not installed here -> row-level asserts carry the same semantics")

# ---------- the transform under test (historical bug FIXED and now locked in) ----------
RATE_SCHEMA = T.StructType([T.StructField("locale", T.StringType()),
                            T.StructField("rate", T.DoubleType())])

def convert_currency(df, rates):
    """amount_cents EUR -> local currency, HALF_UP rounded to 2 decimals.
    The old bug: plain float division + Python round() (banker's rounding) mis-rounded
    x.x5 cases - e.g. 12.085 -> 12.08 instead of 12.09."""
    r = spark.createDataFrame(rates, RATE_SCHEMA)
    return (df.join(r, "locale")
            .withColumn("amount_local",
                        F.expr("CAST(round(amount_cents / 100.0 * rate, 2) AS DECIMAL(18,2))"))
            .select("order_id", "locale", "amount_cents", "rate", "amount_local"))

EXPECTED_SCHEMA = "order_id BIGINT, locale STRING, amount_cents BIGINT, rate DOUBLE, amount_local DECIMAL(18,2)"

# ---------- tests (pytest collects these as test_* functions) ----------
def test_half_up_rounding_and_schema():
    df = spark.createDataFrame([(1, "en-US", 1099),   # 10.99 * 1.10 = 12.089 -> 12.09
                                (2, "ja-JP", 250)],   # 2.50 * 161.80 = 404.50
                               ["order_id", "locale", "amount_cents"])
    out = convert_currency(df, [("en-US", 1.10), ("ja-JP", 161.80)])
    expected = spark.createDataFrame(
        [(1, "en-US", 1099, 1.10, Decimal("12.09")),
         (2, "ja-JP", 250, 161.80, Decimal("404.50"))], EXPECTED_SCHEMA)

    if CHISPA:
        chispa.assert_df_equality(out, expected, ignore_row_order=True, ignore_nullable=True)
        chispa.assert_schema_equality(out.schema, expected.schema, ignore_nullable=True)

    rows = {r["order_id"]: r["amount_local"] for r in out.collect()}
    assert str(rows[1]) == "12.09"      # the HALF_UP case that used to ship as 12.08
    assert str(rows[2]) == "404.50"


def test_no_rate_means_no_row():
    df = spark.createDataFrame([(9, "xx-XX", 100)], ["order_id", "locale", "amount_cents"])
    out = convert_currency(df, [("en-US", 1.10)])
    assert out.count() == 0             # inner join drops unconvertible orders - document it

test_half_up_rounding_and_schema()
test_no_rate_means_no_row()

print("""
[THE TEST PYRAMID FOR DATA - SAY THIS OUT LOUD]
1. Unit (fast, every PR): pure transforms/UDFs on tiny DataFrames - chispa row/schema
   asserts. Hundreds run in minutes.
2. Integration (nightly): real sources/sinks, Delta commits, MERGE logic - a few, against
   staging catalogs.
3. Golden datasets (regression): curated input->expected snapshots; compare schema hash +
   row checksums so unintended drift FAILS the build instead of shipping.
CI runs 1 on every PR; 2+3 on schedule. Tests live with the notebook code in the repo.
""")
print("Q17 OK — HALF_UP rounding regression locked in; schema + drop semantics covered")

## Q18 — Monitoring: Design a Data Observability Stack for 500 Pipelines [HARD]

**Interviewer (Netflix):** Our platform runs 500 pipelines and 3,000 tables, with SLAs owed to external customers. Design the **observability layer** end-to-end — metrics, storage, alerting, on-call, and cost.

1. **Metric hierarchy**: pipeline health (job/task-level) vs data health (freshness, volume, schema, quality) vs cost. Which system tables back each layer?
2. **Architecture**: where metrics land (metrics tables + quality monitors), how alerts route (severity -> channel), and how they link to runbooks. Why alert on **symptoms (SLA breach)** rather than causes (job failed)?
3. **Anti-noise & operations**: alert budgets per team, dedup/flapping control, weekly review. How do you keep 500 pipelines from generating 5,000 alerts a week?

In [0]:
# Solution Q18 — the layered observability blueprint (design question -> design answer)

BLUEPRINT = {
  "L1 PIPELINE HEALTH (is the machinery running?)": """
    Source: orchestration system tables (job-run timelines, task-level events) + native
    retries/repair. Metrics: run success rate, duration p95, queue wait, task retries,
    repair-run frequency per pipeline. Rooted in the orchestrator - that is where retries
    and dependencies already live.""",
  "L2 DATA HEALTH (is the output any good?)": """
    Freshness: last commit ts vs SLA (from DESCRIBE HISTORY / table metrics).
    Volume: row count + bytes vs a trailing 30-day band (robust z / MAD, not raw min-max).
    Schema: schema hash diff vs the data contract (additive vs breaking).
    Quality: nulls, dupes, referential integrity from monitors/expectations.
    Everything is EMITTED AS ROWS into per-table metrics Delta tables: queryable, cheap,
    replayable - and exactly what alerts evaluate.""",
  "L3 COST (can we afford this?)": """
    DBU/spend per pipeline from billing system tables joined to custom tags (team, product).
    Chargeback dashboards + cost-per-refresh trend; alert on UNIT-COST anomalies
    (cost per million rows), not just absolute spend.""",
  "ALERT ROUTING": """
    S1 = customer SLA breached/impending -> PagerDuty + incident channel.
    S2 = internal SLA missed -> team channel, business hours.
    S3 = trend anomaly -> weekly digest.
    Every alert MUST carry an owning team tag and a runbook link - an alert without an
    owner is noise by definition.""",
  "SYMPTOM vs CAUSE": """
    Alert on SYMPTOMS (table is 2h stale / SLA will breach in 30 min) - a pipeline can fail
    50 ways but customers feel only a few symptoms. Causes (task failed, backpressure,
    upstream late) are diagnostic context attached to the symptom page - not separate pages.
    One incident -> one page, no matter how many tasks failed underneath.""",
  "ANTI-NOISE": """
    - Alert BUDGET: each team gets N pages/week; exceeded -> prune the lowest-value alerts.
    - Flapping control: page only on sustained breach (2 consecutive evaluations).
    - Dedup by (table, pipeline) - never by (task, timestamp).
    - Weekly review: top noisiest alerts get fixed, tuned, or deleted.
    Target: <5 actionable pages/week platform-wide. More than that and on-call ignores
    real ones - which is the actual failure mode of observability stacks.""",
  "ROLLOUT": """
    Weeks 1-2: metrics tables + freshness for the top-20 revenue tables. Weeks 3-4:
    volume/schema bands + severity routing. Month 2: quality expectations + cost dashboards.
    Start from the symptoms customers already complain about - not a 3,000-table big bang.""",
}
for k, v in BLUEPRINT.items():
    print(f"### {k}\n{v}\n")

example_queries = """
-- freshness (the SYMPTOM customers feel), from the metrics table the monitors fill
SELECT table_name,
       MAX(committed_at) AS last_commit,
       DATEDIFF(minute, MAX(committed_at), CURRENT_TIMESTAMP()) AS staleness_min,
       MAX(sla_minutes) AS sla_min
FROM ops.table_freshness_metrics
GROUP BY table_name
HAVING DATEDIFF(minute, MAX(committed_at), CURRENT_TIMESTAMP()) > MAX(sla_minutes);

-- volume band breach (MAD-based: robust to outliers, only real incidents page)
SELECT table_name, dt, rows, robust_z
FROM ops.table_volume_metrics
WHERE ABS(robust_z) > 6;
"""
print(example_queries)
print("Q18 OK — three metric layers + symptom-first routing + a noise budget")

## Day 19 — Score Card

Score each question **0-3** after attempting it out loud (before reading the solution):

* **0** — couldn't answer · **1** — needed major hints · **2** — solid with minor gaps · **3** — interview-ready, named the trade-offs unprompted

| Q | Category | Topic | Level | Score (0-3) |
| --- | --- | --- | --- | --- |
| Q1 | SQL Advanced | Multi-touch attribution (first/last/linear/position-based) | ADVANCED | |
| Q2 | SQL Advanced | Fuzzy entity resolution (levenshtein + blocking) | ADVANCED | |
| Q3 | Python | K-way merge of sorted streams (lazy heap generator) | ADVANCED | |
| Q4 | Python | Consistent hashing ring with virtual nodes | HARD | |
| Q5 | PySpark | Arbitrary stateful sessionization | ADVANCED | |
| Q6 | PySpark | Stream-to-stream join, dual watermarks | ADVANCED | |
| Q7 | Spark Optimization | Spark Connect & the UDF serialization tax | ADVANCED | |
| Q8 | Spark Optimization | Sort-merge vs shuffled hash join | HARD | |
| Q9 | ETL | Blue-green expand-contract schema migration | ADVANCED | |
| Q10 | ETL | DEEP CLONE for DR & env seeding | ADVANCED | |
| Q11 | ETL | Checkpoint-safe streaming upgrades | HARD | |
| Q12 | Data Modeling | Event sourcing vs state tables (replayable projections) | ADVANCED | |
| Q13 | Data Modeling | Feature store — point-in-time correctness | ADVANCED | |
| Q14 | Snowflake | Cortex AI functions in GenAI pipelines | ADVANCED | |
| Q15 | Snowflake | Event tables & Snowflake Logging | ADVANCED | |
| Q16 | Monitoring | dbt + Databricks orchestration & incremental models | ADVANCED | |
| Q17 | Monitoring | pytest + chispa pipeline testing | ADVANCED | |
| Q18 | Monitoring | Observability stack design | HARD | |

**Review rules**:

* Any question scoring **< 2** → re-attempt it tomorrow morning BEFORE starting Day 20.
* Two questions **< 2** in the same category → reread that category's Day 15-18 questions too; the gap is foundational.
* Keep every answer under **4 minutes** — length signals lack of judgment, not depth.
* Log today's total: ____ / 54. The trend line across the week matters more than any single day.

> **Tomorrow — Day 20 (Week 3):** deeper system-design judgment — multi-tenant isolation, cost governance at platform scale, and more 2025-2026 trending topics (streaming Lakehouse, GenAI data pipelines). Days 22+ go full FAANG level.